In [18]:
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics.pairwise import haversine_distances

In [19]:
mandis = pd.read_csv("india_mandis_cleaned.csv")

mandis = mandis[["mandi", "lat", "lon"]].drop_duplicates().reset_index(drop=True)

def normalize_mandi(x):
    return (
        str(x).upper()
        .replace(" APMC", "")
        .replace(" MANDI", "")
        .replace("(APMC)", "")
        .replace("(MANDI)", "")
        .replace(",", "")
        .strip()
    )

mandis["mandi_norm"] = mandis["mandi"].apply(normalize_mandi)

mandi_list = mandis["mandi_norm"].tolist()
mandi_idx = {m: i for i, m in enumerate(mandi_list)}

N = len(mandis)
mandis.head()

,mandi,lat,lon,mandi_norm
0,ARAKU VALLEY APMC,18.184897,82.708383,ARAKU VALLEY
1,RAMPACHODVARAM APMC,17.627876,83.158189,RAMPACHODVARAM
2,MADYGULA APMC,17.627876,83.158189,MADYGULA
3,PAYAKARAOPETA APMC,17.627876,83.158189,PAYAKARAOPETA
4,DHARMAVARAM APMC,17.230189,82.227655,DHARMAVARAM


In [20]:
prices = pd.read_csv("enam_prices_cleaned.csv", parse_dates=["date"])
prices = prices[["mandi", "date", "modal_price"]]

prices["mandi_norm"] = prices["mandi"].apply(normalize_mandi)

prices = prices[prices["mandi_norm"].isin(mandi_idx)].copy()
prices["mandi_idx"] = prices["mandi_norm"].map(mandi_idx)
prices.head()

,mandi,date,modal_price,mandi_norm,mandi_idx
4,AGRA,2026-01-05,715,AGRA,2913
5,AGRA,2026-01-06,705,AGRA,2913
6,AGRA,2026-01-07,708,AGRA,2913
7,AGRA,2026-01-08,708,AGRA,2913
8,AGRA,2026-01-09,689,AGRA,2913


In [21]:
dates = np.sort(prices["date"].unique())
T = len(dates)

Y = np.full((T, N), np.nan)

for _, r in prices.iterrows():
    t = np.where(dates == r["date"])[0][0]
    Y[t, r["mandi_idx"]] = r["modal_price"]

Y.shape

(7, 3209)

In [22]:
coords = np.radians(mandis[["lat", "lon"]].values)
dist = haversine_distances(coords, coords) * 6371.0

d0 = 150
A = np.exp(-dist / d0)
np.fill_diagonal(A, 1.0)
A = A / A.sum(axis=1, keepdims=True)

A.shape

(3209, 3209)

In [23]:
ds = xr.open_dataset(
    r"C:\Users\nisch\Desktop\RESEARCH\PRICE_PREDICTION_WORK\data\raw\netcdf\era5_2023_01.nc",
    engine="netcdf4"
)

ds["t2m"] = ds["t2m"] - 273.15
ds["tp"] = ds["tp"] * 1000
ds["ssr"] = ds["ssr"] / 1e6

ds_daily = xr.Dataset({
    "t2m": ds["t2m"].resample(valid_time="1D").mean(),
    "tp": ds["tp"].resample(valid_time="1D").sum(),
    "ssr": ds["ssr"].resample(valid_time="1D").mean()
})

ds_daily

<xarray.Dataset> Size: 34MB
Dimensions:     (latitude: 301, longitude: 301, valid_time: 31)
Coordinates:
  * latitude    (latitude) float64 2kB 38.0 37.9 37.8 37.7 ... 8.3 8.2 8.1 8.0
  * longitude   (longitude) float64 2kB 68.0 68.1 68.2 68.3 ... 97.8 97.9 98.0
  * valid_time  (valid_time) datetime64[ns] 248B 2023-01-01 ... 2023-01-31
    number      int64 8B 0
Data variables:
    t2m         (valid_time, latitude, longitude) float32 11MB 0.7516 ... nan
    tp          (valid_time, latitude, longitude) float32 11MB 0.0457 ... 0.0
    ssr         (valid_time, latitude, longitude) float32 11MB 5.76 ... nan

In [24]:
def build_mask(ds, lat, lon, radius_km=50):
    latg = ds.latitude.values[:, None]
    long = ds.longitude.values[None, :]

    R = 6371.0
    dlat = np.radians(latg - lat)
    dlon = np.radians(long - lon)

    a = (
        np.sin(dlat / 2) ** 2 +
        np.cos(np.radians(lat)) *
        np.cos(np.radians(latg)) *
        np.sin(dlon / 2) ** 2
    )
    dist = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return dist <= radius_km


masks = {
    i: build_mask(ds_daily, r.lat, r.lon)
    for i, r in mandis.iterrows()
}


In [25]:
def get_weather(ds, date, mask):
    date = pd.to_datetime(date).normalize()
    day = ds.sel(valid_time=slice(date, date + pd.Timedelta(days=1)))

    if day.valid_time.size == 0:
        return np.nan

    return day["tp"].where(mask).sum().item()

In [26]:
Z = np.copy(Y)
for t in range(1, T):
    Z[t] = np.where(np.isnan(Y[t]), Y[t-1], Y[t])

In [27]:
def enkf(Y, Z, A, ds, dates, masks,
         n_ens=30, Q=25.0, R=100.0, alpha=0.05):

    T, N = Y.shape

    X = np.zeros((n_ens, N))

    # safe initialization: last observed price per mandi
    for i in range(N):
        valid = Y[:, i][~np.isnan(Y[:, i])]
        if len(valid) == 0:
            X[:, i] = 0.0
        else:
            X[:, i] = valid[0]

    out = np.zeros((T, N))

    for t in range(T):
        weather = np.array([
            get_weather(ds, dates[t], masks[i])
            for i in range(N)
        ])

        for k in range(n_ens):
            noise = np.random.normal(0, np.sqrt(Q), N)
            X[k] = A @ X[k] + alpha * weather + noise

        for i in range(N):
            if not np.isnan(Z[t, i]):
                P = np.var(X[:, i])
                if P > 0:
                    K = P / (P + R)
                    X[:, i] += K * (Z[t, i] - X[:, i])

        out[t] = X.mean(axis=0)

    return out

In [28]:
X_hat = enkf(Y, Z, A, ds_daily, dates, masks)
X_hat.shape

(7, 3209)

In [29]:
forecast = pd.DataFrame(X_hat, index=dates, columns=mandi_list)
forecast.head(), forecast.isna().sum().sum()

(            ARAKU VALLEY  RAMPACHODVARAM  MADYGULA  PAYAKARAOPETA  \
 2026-01-05           NaN             NaN       NaN            NaN   
 2026-01-06           NaN             NaN       NaN            NaN   
 2026-01-07           NaN             NaN       NaN            NaN   
 2026-01-08           NaN             NaN       NaN            NaN   
 2026-01-09           NaN             NaN       NaN            NaN   
 
             DHARMAVARAM  ANGALLU  VALMIKIPURAM  VEMURU  BANGARUPALYAM  \
 2026-01-05          NaN      NaN           NaN     NaN            NaN   
 2026-01-06          NaN      NaN           NaN     NaN            NaN   
 2026-01-07          NaN      NaN           NaN     NaN            NaN   
 2026-01-08          NaN      NaN           NaN     NaN            NaN   
 2026-01-09          NaN      NaN           NaN     NaN            NaN   
 
             CHITTOR(RYTHU BAZAR)  ...  BURDWAN  KATWA  MEMARI  BALARAMPUR  \
 2026-01-05                   NaN  ...      NaN    NaN

In [ ]:
m = normalize_mandi("GURGAON")
i = mandi_idx[m]

pd.DataFrame({
    "actual": Y[:, i],
    "forecast": X_hat[:, i]
}).head(10)

,actual,forecast
0,766.0,NaN
1,655.0,NaN
2,655.0,NaN
3,655.0,NaN
4,655.0,NaN
5,988.0,NaN
6,766.0,NaN
